### MCP Server setup

Uses `mcp` SDK directly (compatible with `mcp>=2.0.0`) — no `langchain-mcp-adapters` required.

- DuckDuckGo MCP server (downloaded automatically via npx on first run):
```shell
npx -y duckduckgo-mcp-server
```

- Check LLM:
```shell
ollama show llama3.2
```

In [ ]:
# !pip install mcp langgraph langchain-core langchain-ollama pydantic

In [5]:
from typing import Any, Optional, Literal

from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters
from langchain_core.tools import StructuredTool
from pydantic import create_model
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent


model = ChatOllama(
    model="llama3.2",
    temperature=0
)


def mcp_tool_to_langchain(mcp_tool, session: ClientSession) -> StructuredTool:
    """Wrap an MCP tool as a LangChain StructuredTool using mcp SDK >= 2.0.

    - Enum fields use Literal so the LLM only picks valid values.
    - Optional fields use their schema default so the LLM doesn't have to supply them.
    - None values are stripped before calling the MCP server so it applies its own defaults.
    """
    properties = mcp_tool.input_schema.get("properties", {})
    required = set(mcp_tool.input_schema.get("required", []))

    fields = {}
    for name, schema in properties.items():
        t = schema.get("type")
        enum_vals = schema.get("enum")
        default = schema.get("default")
        is_required = name in required

        if enum_vals:
            annotation = Literal[tuple(enum_vals)]
        elif t == "integer":
            annotation = int
        elif t == "number":
            annotation = float
        elif t == "boolean":
            annotation = bool
        else:
            annotation = str

        if is_required:
            fields[name] = (annotation, ...)
        elif default is not None:
            fields[name] = (Optional[annotation], default)
        else:
            fields[name] = (Optional[annotation], None)

    ArgsModel = create_model(f"{mcp_tool.name}_args", **fields)

    async def call_tool(**kwargs: Any) -> str:
        # Strip None values — let the MCP server apply its own defaults
        clean_args = {k: v for k, v in kwargs.items() if v is not None}
        result = await session.call_tool(mcp_tool.name, arguments=clean_args)
        return "\n".join(
            block.text for block in result.content if hasattr(block, "text")
        )

    return StructuredTool.from_function(
        coroutine=call_tool,
        name=mcp_tool.name,
        description=mcp_tool.description or "",
        args_schema=ArgsModel,
    )


server_params = StdioServerParameters(
    command="npx",
    args=["-y", "duckduckgo-mcp-server"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        response = await session.list_tools()
        tools = [mcp_tool_to_langchain(t, session) for t in response.tools]

        print(f"Tools loaded: {[t.name for t in tools]}")

        agent = create_react_agent(
            model=model,
            tools=tools,
            prompt="""
You are a Solution Architect.
You work on real problems and provide solutions using latest technologies, Cloud and AI.
You document the solution and update it.
Use web search to find the solution.
Your solution should cover edge cases.
Your solution should support large scale.
Your solution should cover all the aspects of architecture.
"""
        )

        result = await agent.ainvoke({
            "messages": [{
                "role": "user",
                "content": """
Requirements:
Design a system which can enable users to find a product in the inventory,
search for the product using natural language,
fetch a particular model based on user query.
"""
            }]
        })

        print(result["messages"][-1].content)

Tools loaded: ['duckduckgo_web_search']


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_88598/3056075298.py:84: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


To design a system that enables users to find a product in the inventory, search for the product using natural language, and fetch a particular model based on user query, I propose the following solution:

**System Architecture**

The proposed system will consist of three main components:

1. **Natural Language Processing (NLP) Module**: This module will be responsible for processing user queries and converting them into a format that can be understood by the inventory management system.
2. **Inventory Management System**: This system will store information about all products in the inventory, including their models, descriptions, prices, and other relevant details.
3. **Search Engine**: This component will be used to search for products based on user queries.

**NLP Module**

The NLP module will use a library such as NLTK or spaCy to perform natural language processing on user queries. It will extract relevant information from the query, such as product names, descriptions, and keywor